In [ ]:
# Ensure you have at least 2GB free space in Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Project setup
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())

# Install dependencies
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics

In [ ]:
# Run this everytime you update something in the repo!
REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

# if project directory is empty clone the repo, otherwise pull the latest changes
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("Working in:", os.getcwd())

In [ ]:
import random, shutil, zipfile
import subprocess
from pathlib import Path
from PIL import Image
import glob
import importlib
  


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, average_precision_score, hamming_loss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import pytorch_lightning as L
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from pytorch_lightning.loggers import CSVLogger
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

import torchmetrics
import re 

# utility functions
import utils


L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

Download dataset

In [ ]:
"Download the UCMerced Land Use dataset if not already present. "
"The dataset will be saved in the 'ucmdata' directory. "
if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')

    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
        zip_ref.extractall('UCMImages')

    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
UCM_images_path = "/ucmdata/Images/"
Multilabels_path = "/ucmdata/LandUse_Multilabeled.txt"

print(os.listdir('.'))

# After git pull, reload utils to get the latest changes without restarting the notebook
importlib.reload(utils)

Data split and augmentation

In [ ]:
# Full dataloader test 
# Can take a long time (4.5min on Colab GPU T4)
from utils import build_dataloaders

train_loader, val_loader, test_loader, classes, pos_w = build_dataloaders(
    root_dir = "ucmdata",
    label_file = "LandUse_Multilabeled.txt",
    image_size = (224, 224),
    batch_size = 32,
    num_workers = 2,
    val_frac = 0.15,
    test_frac = 0.15,
    seed = 42,
    image_ext = ".tif",
)

train_labels = torch.cat([labels for _, labels in train_loader]).numpy()
val_labels = torch.cat([labels for _, labels in val_loader]).numpy()
test_labels = torch.cat([labels for _, labels in test_loader]).numpy()

# Create a dataframe and export to csv that appends the class frequencies
import numpy as np
distribution_df = pd.DataFrame({
    "class": classes,
    "train_freq": train_labels.sum(axis=0),
    "val_freq": val_labels.sum(axis=0),
    "test_freq": test_labels.sum(axis=0),
})
distribution_df.to_csv("outputs/Balanced_class_distribution.csv", index=False)


Initialize WANDB

Hyperparameters 

In [ ]:
# Define hyperparameters
PRETRAINED_MODEL = "vit_b_16"
NUM_CLASSES = len(classes)
MAX_EPOCHS = 25
EARLYSTOPPING_EPOCHS = 5
LR = 3e-5
WEIGHT_DECAY = 5e-2
run_count = 1


Load and train model

In [ ]:
def build_vit_b16(num_classes):
    weights = tvm.ViT_B_16_Weights.IMAGENET1K_V1
    model = tvm.vit_b_16(weights=weights)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

backbone = build_vit_b16(num_classes=NUM_CLASSES)
n_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"ViT-B/16 trainable parameters: {n_params:,}  |  output logits: {NUM_CLASSES}")

Training

In [ ]:
from utils import LightningModuleMultilabel 

lit_model = LightningModuleMultilabel(model=backbone, num_classes=NUM_CLASSES,
                                      lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS, pos_weight=pos_w)
# Callbacks 
checkpoint_cb = ModelCheckpoint(
    dirpath="outputs/checkpoints",
    filename="vit_b16-multilabel-best-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1, save_weights_only=True,
)

early_stopping = EarlyStopping(
            monitor="val_loss",  
            patience=EARLYSTOPPING_EPOCHS,        # Number of epochs with no improvement
            mode="min",        # Minimize val loss
            verbose=True
            )

# Logger to plot later
csv_logger = CSVLogger(save_dir="outputs/logs", name="vit_b16_multilabel")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    callbacks=[early_stopping, checkpoint_cb],
    logger=csv_logger,
    log_every_n_steps=10,
)
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)


Final classification on test data

In [ ]:
# Evaluate using the best checkpoint
trainer.test(lit_model, dataloaders=test_loader, ckpt_path="best")

best_path = checkpoint_cb.best_model_path
print(f"Best checkpoint: {best_path}")
lit_model = LightningModuleMultilabel.load_from_checkpoint(best_path, model=lit_model.model)

In [ ]:

# Collect probs, preds and ground-truth for downstream analysis
preds_out = trainer.predict(lit_model, dataloaders=test_loader)
test_probs  = torch.cat([b["probs"]  for b in preds_out], dim=0).cpu().numpy()
test_preds  = torch.cat([b["preds"]  for b in preds_out], dim=0).cpu().numpy()
test_labels = torch.cat([b["labels"] for b in preds_out], dim=0).cpu().numpy()


In [ ]:
from utils import compute_test_metrics

# Usage
metrics = compute_test_metrics(test_preds, test_labels, test_probs)

# Print metrics
print(f"\nTest accuracy   : {metrics['accuracy']:.4f}")
print(f"Test macro F1   : {metrics['macro_f1']:.4f}")
print(f"Test micro F1   : {metrics['micro_f1']:.4f}")
print(f"Test samples F1 : {metrics['samples_f1']:.4f}")
print(f"Test macro mAP  : {metrics['macro_map']:.4f}")
print(f"Hamming loss    : {metrics['hamming_loss']:.4f}")
print(f"Exact-match acc : {metrics['subset_acc']:.4f}")

# Export and append to a csv of the metrics f1 score of many other models and runs for comparison and plotting later
from utils import append_metrics_to_csv

append_metrics_to_csv(metrics, model_name="ML_ViT_b16_Balanced")



Visualize

In [ ]:
# Usage
from utils import plot_training_curves
plot_training_curves(csv_logger, model_name="ViT-B/16 multilabel", 
                     save_path=None)

In [ ]:
from utils import plot_per_class_metrics

# Usage
fig, ax, summary_df = plot_per_class_metrics(
    test_labels=test_labels,
    test_preds=test_preds,
    test_probs=test_probs,
    classes=classes,
    macro_f1=metrics['macro_f1'],
    macro_map=metrics['macro_map'],
    model_name="ViT-B/16 multilabel",
    save_path="outputs/vit_b16_multilabel_per_class.png",
    csv_output="outputs/vit_b16_multilabel_per_class.csv"
)

Examples of images

In [ ]:

from utils import plot_prediction_grid
fig = plot_prediction_grid(
    test_preds=test_preds,
    test_labels=test_labels,
    test_probs=test_probs,
    classes=classes,
    test_loader=test_loader,
    root_dir="ucmdata",
    label_file="LandUse_Multilabeled.txt",
    n_show=9,
    seed=4,
    save_path="outputs/vit_b16_multilabel_predictions_grid.png"
)